# ⏳ Notebook 3 — Limits of Hinted Handoff: TTL, Coordinator Crashes, and Sloppy Quorum

Hinted handoff is a lightweight patch, **not** a full repair system. This notebook makes the failure modes concrete so you know when it helps and when it doesn't.

We will:

1. Add a **TTL** (time-to-live) to hints and watch them expire — just like Cassandra's `max_hint_window_in_ms` (default: **3 hours**).
2. Kill the coordinator and see that hints stored only in memory are lost.
3. Introduce **sloppy quorum**: when the *real* replica is down, the coordinator writes to a *temporary* replica instead and leaves a hint for hand-off.
4. Summarize when you still need **anti-entropy / Merkle-tree repair** on top.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/hinted-handoff
uv sync
```

Select the `.venv` kernel. `Cmd+Shift+P` → **Reload Window** if it doesn't appear.

## 1️⃣ Hint TTL — hints don't live forever

Hints take memory and disk on the coordinator. If `r3` stays down for a week, the coordinator cannot buffer every write forever.
Real systems cap the *hint window*: after N hours, the coordinator **stops** generating new hints and eventually **expires** old ones. At that point, only anti-entropy repair can bring `r3` back into sync.

We'll use a simple integer clock (ticks) instead of wall-clock time so the notebook is deterministic and fast.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Deque, Dict, List, Tuple

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str, int]] = field(default_factory=dict)
    def write(self, k, v, ts):
        if not self.up:
            return False
        cur = self.data.get(k)
        if cur is None or ts >= cur[1]:
            self.data[k] = (v, ts)
        return True


@dataclass
class Hint:
    target: str
    key: str
    value: str
    ts: int          # write time (logical)
    created_at: int  # when the hint was stored (logical)


class Coordinator:
    def __init__(self, replicas: List[Replica], hint_ttl: int = 10):
        self.replicas = replicas
        self.hints: Dict[str, Deque[Hint]] = defaultdict(deque)
        self.hint_ttl = hint_ttl          # how many ticks a hint is kept
        self.now = 0                      # logical clock
        self.dropped = 0                  # stats: how many hints we threw away

    def tick(self, n: int = 1) -> None:
        self.now += n

    def _expire(self) -> None:
        for q in self.hints.values():
            while q and self.now - q[0].created_at > self.hint_ttl:
                exp = q.popleft()
                self.dropped += 1
                print(f'  🗑 dropping expired hint for {exp.target}: '
                      f'{exp.key}={exp.value} (age={self.now - exp.created_at} > ttl={self.hint_ttl})')

    def write(self, key: str, value: str) -> None:
        self.tick()
        self._expire()
        ts = self.now
        for r in self.replicas:
            if not r.write(key, value, ts):
                self.hints[r.name].append(Hint(r.name, key, value, ts, self.now))

    def deliver_hints(self) -> None:
        self._expire()
        for r in self.replicas:
            if not r.up:
                continue
            delivered = 0
            while self.hints[r.name]:
                h = self.hints[r.name].popleft()
                r.write(h.key, h.value, h.ts)
                delivered += 1
            if delivered:
                print(f'  ✉ delivered {delivered} hint(s) to {r.name}')


In [ ]:
replicas = [Replica('r1'), Replica('r2'), Replica('r3')]
coord = Coordinator(replicas, hint_ttl=5)  # hints live 5 ticks

replicas[2].up = False
coord.write('k1', 'v1')
coord.write('k2', 'v2')

# r3 stays down for a long time — simulate 10 idle ticks going by,
# with one write per tick to trigger the expiry check.
for _ in range(10):
    coord.write('noise', 'noise')

print('\nr3 finally recovers — too late:')
replicas[2].up = True
coord.deliver_hints()

print(f'\nhints dropped due to TTL: {coord.dropped}')
print(f'r3 data: {replicas[2].data}')


### 🧐 What happened

- The hints for `k1` and `k2` aged past the TTL and were **discarded**.
- When `r3` came back, the coordinator had *nothing* to hand off for those keys.
- `r3` is now **silently stale** for `k1`, `k2` — only a full repair can fix it.

**Takeaway:** hinted handoff covers *brief* outages (minutes to a few hours). For longer outages, plan on anti-entropy repair (see the [`merkle-trees`](../../merkle-trees/) lab).

## 2️⃣ What if the coordinator itself dies?

Hints stored only in memory evaporate when the coordinator crashes. 
Production systems persist hints to **disk** (Cassandra writes them to a dedicated hints directory). Let's simulate a crash to see what is at stake.

In [ ]:
# Re-run a clean scenario, then 'crash' the coordinator before hints are delivered.
replicas = [Replica('r1'), Replica('r2'), Replica('r3')]
coord = Coordinator(replicas, hint_ttl=999)

replicas[2].up = False
coord.write('order:42', 'paid')
coord.write('order:43', 'paid')
print(f'pending hints: {len(coord.hints["r3"])}')

# 💥 coordinator crashes — we just throw away the object.
del coord
coord = Coordinator(replicas, hint_ttl=999)  # a fresh coordinator starts with no memory

replicas[2].up = True
coord.deliver_hints()
print(f'r3 data after "recovery": {replicas[2].data}  # 🕳 writes lost!')


**Lesson:** in real systems, persist hints to durable storage (disk, or a replicated hint log). Otherwise a coordinator crash == hinted handoff failure.

## 3️⃣ Sloppy Quorum — write *somewhere* instead of failing

What if you need to return success *right now* but the primary replica is down? 
**Sloppy quorum** (from the Dynamo paper) says: write the copy to **any healthy node**, and attach a hint telling that node to hand it off to the real replica later.

In strict quorum: a write needs `W` confirmations from the *designated* replicas. 
In sloppy quorum: a write needs `W` confirmations from *any* live nodes, with hints for the ones that should eventually own the data.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

@dataclass
class Node:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str,int]] = field(default_factory=dict)
    handoffs: List[Tuple[str,str,str,int]] = field(default_factory=list)  # (target, k, v, ts)
    def write(self, k, v, ts):
        if not self.up: return False
        cur = self.data.get(k)
        if cur is None or ts >= cur[1]:
            self.data[k] = (v, ts)
        return True

class SloppyCoordinator:
    def __init__(self, preference_list: List[Node], standbys: List[Node], W: int = 2):
        self.preference_list = preference_list   # the 'real' replicas for this key
        self.standbys = standbys                 # spare nodes that can absorb writes
        self.W = W
        self.clock = 0

    def write(self, key, value):
        self.clock += 1
        ts = self.clock
        acks = 0
        for r in self.preference_list:
            if r.write(key, value, ts):
                acks += 1
                print(f'  ✅ {r.name} (preferred) acked')
            else:
                # Hand the write to a live standby and mark it for hand-off to r.
                standby = next((s for s in self.standbys if s.up), None)
                if standby is None:
                    print(f'  ❌ no standby available for {r.name}')
                    continue
                standby.write(key, value, ts)
                standby.handoffs.append((r.name, key, value, ts))
                acks += 1
                print(f'  🛟 {standby.name} absorbed write meant for {r.name} (hinted)')
        ok = acks >= self.W
        print(f'  → {"WRITE OK" if ok else "WRITE FAIL"} (acks={acks}, W={self.W})')
        return ok

    def hand_off(self):
        for s in self.standbys:
            remaining = []
            for target_name, k, v, ts in s.handoffs:
                target = next(r for r in self.preference_list if r.name == target_name)
                if target.up and target.write(k, v, ts):
                    print(f'  ✉ {s.name} → {target.name}: {k}={v}')
                else:
                    remaining.append((target_name, k, v, ts))
            s.handoffs = remaining

preferred = [Node('r1'), Node('r2'), Node('r3')]
standbys  = [Node('spare-A'), Node('spare-B')]
coord = SloppyCoordinator(preferred, standbys, W=2)

preferred[1].up = False        # r2 is down
preferred[2].up = False        # r3 is down
print('--- write with strict quorum would FAIL (only r1 is up) ---')
coord.write('cart:7', 'item=42')  # sloppy quorum rescues us

print('\n--- r2, r3 recover; hand off from standbys ---')
preferred[1].up = True
preferred[2].up = True
coord.hand_off()

for n in preferred + standbys:
    print(f'  {n.name}: data={n.data}, pending_handoffs={n.handoffs}')


### 🔑 Strict vs sloppy quorum — side by side

| | Strict quorum | Sloppy quorum + hinted handoff |
|---|---|---|
| If designated replicas are down | write **fails** (unavailable) | write **succeeds** on a standby |
| Availability under partial outage | lower | higher |
| Risk | none | standby could crash before hand-off → data loss |
| Example system | some strong-consistency configs | DynamoDB, Cassandra (default) |

## 4️⃣ When hinted handoff is **not** enough

Use hinted handoff **alongside** — never instead of — other repair mechanisms:

- 🧵 **Anti-entropy / Merkle-tree repair** — fixes the long tail (outages longer than the hint window, dropped hints, coordinator crashes). See [`../merkle-trees/`](../../merkle-trees/).
- 🧪 **Read-repair** — opportunistically syncs a replica when its stale value is seen during a read. See [`../read-repair/`](../../read-repair/).
- 🧭 **Gossip / failure detection** — without knowing *who* is up, the coordinator can't choose standbys wisely. See [`../gossip-protocol/`](../../gossip-protocol/) and [`../phi-accrual-failure-detection/`](../../phi-accrual-failure-detection/).

Hinted handoff is **cheap + targeted**. The other three are **general + expensive**. Real systems run all of them in concert.

## 🌍 Real-world examples

- **Apache Cassandra** — `max_hint_window_in_ms` (default 3 h), hints persisted under `/var/lib/cassandra/hints/`. After the window expires, operators run `nodetool repair`.
- **Amazon DynamoDB (Dynamo paper)** — canonical description of hinted handoff + sloppy quorum. Any healthy node can absorb a write and hand it off when the preferred node returns.
- **ScyllaDB** — Cassandra-compatible; same hint log concept, heavily optimized.
- **Riak** — keeps hints in a vnode's "handoff" directory; also used for cluster rebalancing.

## ✅ Checklist for production

- [ ] Persist hints to disk, not just memory.
- [ ] Cap the hint window (TTL) so a long outage doesn't blow up the coordinator.
- [ ] Rate-limit hint delivery so you don't overwhelm the recovering node.
- [ ] Pair with anti-entropy repair for anything older than the hint window.
- [ ] Use timestamps / version vectors so stale hints never clobber fresh writes.
- [ ] Monitor pending hints per target — a growing queue is an early warning sign.

## 🧠 Recap

1. Without hinted handoff, a replica that misses writes stays stale until the next full repair.
2. With hinted handoff, the coordinator keeps a small per-target FIFO of missed writes and replays them on recovery — **cheap, fast, targeted**.
3. Timestamps keep replays safe when newer writes raced ahead of the hint.
4. Hints have a **TTL** and a **single point of failure** (the coordinator), so you still need anti-entropy repair for the long tail.
5. Sloppy quorum extends the trick to *accepting* writes during outages by temporarily storing them on standbys.